# Lección 7.3 — Estrategias de búsqueda: cuadrícula, aleatoria, bayesiana y *successive halving*

> **Módulo 7 · Regularización y ajuste de hiperparámetros** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 1.2 (parámetro vs. hiperparámetro), Lección 4.6 (validación cruzada), Lección 5.5 (SVM), Lecciones 7.1–7.2

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía las cuatro grandes estrategias de búsqueda de hiperparámetros.
- Usar `GridSearchCV`, `RandomizedSearchCV` y `HalvingRandomSearchCV` de scikit-learn y leer sus resultados.
- Explicar **por qué la búsqueda aleatoria suele ganarle a la cuadrícula** con el mismo presupuesto.
- Entender la idea de la **búsqueda bayesiana** (Optuna) y cuándo vale la pena.
- Reconocer por qué `best_score_` es **optimista** y cómo evitar engañarte.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

> 💡 `n_jobs=-1` le dice a scikit-learn que use **todos los núcleos** de tu computadora para probar varias configuraciones a la vez. Si tu equipo se pone lento, cámbialo por `n_jobs=1`.

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform, randint
from sklearn.experimental import enable_halving_search_cv   # activa las búsquedas "halving" (aún experimentales)
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV, HalvingRandomSearchCV,
                                     train_test_split, cross_val_score)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.datasets import load_digits, make_classification

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: buscar petróleo con un presupuesto

Una empresa tiene un terreno enorme y dinero para perforar **solo 16 pozos**. Cada pozo es caro. ¿Dónde perfora?

| Estrategia | Cómo perfora | En ML se llama |
|---|---|---|
| 🔲 **Cuadrícula** | Divide el terreno en una malla de 4 × 4 y perfora en cada cruce | *Grid search* |
| 🎲 **Al azar** | Lanza 16 dardos al mapa y perfora donde caigan | *Random search* |
| 🧠 **Geólogo** | Perfora unos pocos pozos, **aprende** de lo que encuentra y decide dónde perforar después | Búsqueda **bayesiana** (Optuna) |
| 🏆 **Torneo** | Hace sondeos **baratos y poco profundos** en muchos sitios; solo perfora a fondo los más prometedores | *Successive halving* |

En ML, el **terreno** es el espacio de hiperparámetros (todas las combinaciones posibles), cada **pozo** es entrenar y validar un modelo con una configuración, y el **petróleo** es el puntaje de validación cruzada.

### ✋ Ejercicio 1 — ¿Cuánto cuesta un pozo?
Quieres probar una SVM (Lección 5.5) con:
- `C` en 4 valores,
- `gamma` en 5 valores,
- y evaluar cada combinación con **validación cruzada de 5 pliegues**.

¿Cuántos modelos tendrá que entrenar la búsqueda en cuadrícula? Si cada uno tarda 2 segundos, ¿cuánto tarda en total? ¿Y si añades un tercer hiperparámetro con 4 valores?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- 4 × 5 = **20 combinaciones** × 5 pliegues = **100 modelos** → unos **200 segundos** (≈ 3 min).
- Con un tercer hiperparámetro de 4 valores: 20 × 4 = 80 combinaciones × 5 = **400 modelos** → ≈ 13 min.

Cada hiperparámetro nuevo **multiplica** el costo. Esta es la gran debilidad de la cuadrícula: con 5 o 6 hiperparámetros se vuelve imposible. Se le llama la **explosión combinatoria**.
</details>

---
## 2. 📖 Búsqueda en cuadrícula: `GridSearchCV`

La más sencilla: le das una lista de valores para cada hiperparámetro y prueba **todas** las combinaciones, cada una con validación cruzada. Al final:
- `best_params_` → la mejor combinación,
- `best_score_` → su puntaje medio en validación cruzada,
- `best_estimator_` → el modelo **reentrenado** con esa combinación sobre todo el entrenamiento, listo para usar,
- `cv_results_` → la tabla con **todos** los resultados.

Datos: imágenes de **dígitos escritos a mano** (8 × 8 píxeles) y una SVM.

> 📝 Fíjate en que los valores van en **escala logarítmica** (0.01, 0.1, 1, 10): como viste con `alpha` en la Lección 7.1, el efecto de estas perillas se nota por órdenes de magnitud.

In [ ]:
X_d, y_d = load_digits(return_X_y=True)
X_d_tr, X_d_te, y_d_tr, y_d_te = train_test_split(X_d, y_d, test_size=0.25, stratify=y_d, random_state=0)

rejilla = {"C":     [0.01, 0.1, 1, 10],
           "gamma": [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]}

inicio = time.time()
grid = GridSearchCV(SVC(), rejilla, cv=5, n_jobs=-1).fit(X_d_tr, y_d_tr)
print(f"Probó {len(grid.cv_results_['params'])} combinaciones × 5 pliegues en {time.time() - inicio:.1f} s")
print("Mejores hiperparámetros:", grid.best_params_)
print(f"Mejor exactitud en validación cruzada: {grid.best_score_:.3f}")

### 🔮 Ejercicio 2 — Lee el mapa
La celda siguiente dibuja el **mapa de calor** con la exactitud de cada combinación. Antes de ejecutarla: ¿crees que habrá una zona amplia de buenos resultados o un único punto brillante? ¿Y qué pasará con `gamma` muy grande?

✍️ *Mi predicción:*

In [ ]:
tabla = (pd.DataFrame(grid.cv_results_["params"])
           .assign(exactitud=grid.cv_results_["mean_test_score"])
           .pivot(index="C", columns="gamma", values="exactitud"))

plt.imshow(tabla, cmap="viridis", vmin=0, vmax=1)
plt.xticks(range(len(tabla.columns)), tabla.columns); plt.yticks(range(len(tabla.index)), tabla.index)
plt.xlabel("gamma"); plt.ylabel("C"); plt.colorbar(label="exactitud CV")
for i in range(tabla.shape[0]):
    for j in range(tabla.shape[1]):
        plt.text(j, i, f"{tabla.iloc[i, j]:.2f}", ha="center", va="center",
                 color="w" if tabla.iloc[i, j] < 0.6 else "k")
plt.title("Exactitud de cada combinación"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Hay una **zona** de buenos resultados (≈ 0.96–0.99), no un solo punto: una "cresta" en diagonal donde `C` y `gamma` se compensan. El mejor es `C = 1`, `gamma = 0.001` (≈ 0.988).

Con `gamma` muy grande (0.1) todo se hunde (≈ 0.10–0.11, ¡como adivinar al azar entre 10 dígitos!): cada imagen solo se parece a sí misma, sobreajuste extremo. Con `C` muy pequeño (0.01) también falla: demasiada regularización (Lección 7.1).

Moraleja: las buenas configuraciones suelen formar **regiones**. No necesitas el punto exacto; necesitas caer **dentro** de la región buena.
</details>

### ✋ Ejercicio 3 — El mejor está en el borde
Un compañero hizo esta otra búsqueda. Ejecútala y mira dónde quedó el mejor valor de `C`. ¿Te fiarías del resultado? ¿Qué harías?

In [ ]:
grid_borde = GridSearchCV(SVC(gamma=1e-4), {"C": [0.001, 0.01, 0.1]}, cv=5, n_jobs=-1).fit(X_d_tr, y_d_tr)
for c, s in zip([0.001, 0.01, 0.1], grid_borde.cv_results_["mean_test_score"]):
    print(f"C={c:<6} exactitud CV = {s:.3f}")
print("Mejor:", grid_borde.best_params_)

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

El mejor es `C = 0.1`, **el valor más grande** de la lista, y la exactitud seguía subiendo (0.23 → 0.23 → 0.87). Cuando el ganador queda **en el borde**, es muy probable que el verdadero mejor esté **fuera** del rango que probaste. En el mapa de antes viste que con `gamma = 1e-4` la exactitud sigue subiendo con `C = 1` y `C = 10` (≈ 0.97–0.99).

Regla: **si el mejor queda en el borde, amplía el rango en esa dirección** y vuelve a buscar.
</details>

---
## 3. 📖 Búsqueda aleatoria: por qué el azar gana

`RandomizedSearchCV` no prueba una lista fija: **sortea** `n_iter` combinaciones a partir de **distribuciones** que tú defines (por ejemplo, "cualquier `C` entre 0.01 y 100, en escala logarítmica").

¿Por qué funcionaría mejor sortear que ser ordenado? Por un hecho que se repite en casi todos los modelos: **no todos los hiperparámetros importan lo mismo** (lo verás a fondo en la Lección 7.4). A menudo uno o dos deciden casi todo.

Imagina un terreno donde el petróleo depende **casi solo** de la posición este-oeste (el hiperparámetro importante) y casi nada de la norte-sur (el irrelevante). Comparemos 9 pozos en cuadrícula contra 9 al azar.

### 🔮 Ejercicio 4 — ¿Cuántos valores distintos prueba cada una?
Con 9 pozos en una cuadrícula de 3 × 3, ¿cuántos valores **distintos** del hiperparámetro importante se prueban? ¿Y con 9 pozos al azar?

✍️ *Mi predicción:*

In [ ]:
def puntaje(importante, irrelevante):
    # el "petróleo": depende casi solo del hiperparámetro importante (mejor valor ≈ 0.73)
    return np.exp(-(importante - 0.73) ** 2 / 0.03) + 0.05 * irrelevante

valores = [0.1, 0.5, 0.9]
pozos_grid = np.array([(a, b) for a in valores for b in valores])
pozos_azar = np.random.default_rng(3).uniform(0, 1, size=(9, 2))

a, b = np.meshgrid(np.linspace(0, 1, 200), np.linspace(0, 1, 200))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, pozos, nombre in [(axes[0], pozos_grid, "Cuadrícula 3 × 3"), (axes[1], pozos_azar, "9 pozos al azar")]:
    ax.contourf(a, b, puntaje(a, b), levels=20, cmap="YlOrBr")
    ax.scatter(pozos[:, 0], pozos[:, 1], c="k", s=60)
    ax.plot(pozos[:, 0], np.full(9, -0.04), "|", color="r", ms=20, clip_on=False)  # proyección en el eje importante
    mejor = max(puntaje(p[0], p[1]) for p in pozos)
    ax.set_title(f"{nombre}\nvalores distintos del importante: {len(np.unique(pozos[:, 0]))} · mejor puntaje: {mejor:.2f}")
    ax.set_xlabel("hiperparámetro IMPORTANTE →"); ax.set_ylabel("hiperparámetro irrelevante")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Cuadrícula:** solo **3** valores distintos del importante (0.1, 0.5 y 0.9). Las otras 6 pruebas "repiten" esos valores cambiando únicamente el irrelevante: **pozos desperdiciados**. Ninguno cae cerca de 0.73, así que el mejor puntaje es pobre (≈ 0.43).
- **Al azar:** **9** valores distintos del importante. Alguno cae cerca de la zona buena y el mejor puntaje es mucho más alto (≈ 1.05).

Con el **mismo presupuesto**, la búsqueda aleatoria explora mucho mejor las dimensiones que importan. Como no sabes de antemano cuáles son, sortear es una apuesta segura. Esta idea viene de un artículo famoso de Bergstra y Bengio (2012).
</details>

### ✋ ¿Fue suerte?
Una sola tirada de dados no prueba nada. Repitamos el experimento **1,000 veces** con distintas semillas y contemos cuántas veces gana cada estrategia.

In [ ]:
mejor_grid = max(puntaje(p[0], p[1]) for p in pozos_grid)
victorias_azar = 0
for semilla in range(1000):
    pozos = np.random.default_rng(semilla).uniform(0, 1, size=(9, 2))
    victorias_azar += max(puntaje(p[0], p[1]) for p in pozos) > mejor_grid
print(f"La búsqueda aleatoria encontró algo mejor que la cuadrícula en {victorias_azar / 10:.0f} % de las repeticiones")

<details><summary><b>👀 Ver respuesta</b></summary>

La búsqueda aleatoria le gana a la cuadrícula en **≈ 98 %** de las repeticiones. No es suerte: es que prueba más valores del hiperparámetro que importa. (Si la cuadrícula hubiera tenido la suerte de incluir un valor cercano a 0.73, el resultado cambiaría; pero en la vida real no sabes dónde está el mejor valor).
</details>

### 📖 `RandomizedSearchCV` en la práctica
En lugar de listas, usas **distribuciones**:
- `loguniform(0.01, 100)` → cualquier número entre 0.01 y 100, con la misma probabilidad para cada orden de magnitud. Ideal para `C`, `gamma`, `alpha`, `learning_rate`.
- `randint(1, 20)` → un entero entre 1 y 19. Ideal para `max_depth`, `min_samples_leaf`.
- Una lista (`["gini", "entropy"]`) → elige uno de esos valores.

Y decides el presupuesto con `n_iter`. Mismos datos de dígitos; le damos **el mismo presupuesto** que a la cuadrícula: 20 combinaciones.

In [ ]:
distribuciones = {"C":     loguniform(1e-2, 1e2),
                  "gamma": loguniform(1e-5, 1e-1)}

inicio = time.time()
azar = RandomizedSearchCV(SVC(), distribuciones, n_iter=20, cv=5, n_jobs=-1, random_state=0).fit(X_d_tr, y_d_tr)
print(f"Probó 20 combinaciones en {time.time() - inicio:.1f} s")
print("Mejores:", {k: round(float(v), 5) for k, v in azar.best_params_.items()})
print(f"Mejor exactitud CV: {azar.best_score_:.3f}   (cuadrícula: {grid.best_score_:.3f})")

### ✏️ Ejercicio 5 — El presupuesto
Cambia `N_ITER` (5, 10, 40) y vuelve a ejecutar. ¿Cómo cambian el tiempo y la mejor exactitud?

In [ ]:
N_ITER = 5     # ✏️ prueba 5, 10, 40

inicio = time.time()
azar_n = RandomizedSearchCV(SVC(), distribuciones, n_iter=N_ITER, cv=5, n_jobs=-1, random_state=0).fit(X_d_tr, y_d_tr)
print(f"n_iter={N_ITER}: mejor exactitud CV = {azar_n.best_score_:.3f} · {time.time() - inicio:.1f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

Con 20 combinaciones, la búsqueda aleatoria ya **empata** con la cuadrícula (≈ 0.988). ¡Con solo `N_ITER = 5` llega a ≈ 0.987! Con 10 o 40 queda en ≈ 0.988: el resultado ya no mejora, mientras el tiempo crece en proporción al número de intentos.

Esto se llama **rendimientos decrecientes**: los primeros intentos encuentran la región buena; los siguientes solo pulen decimales. Una gran ventaja de la búsqueda aleatoria es que **tú controlas el presupuesto** con un solo número, sin importar cuántos hiperparámetros tengas.
</details>

---
## 4. 📖 *Successive halving*: el torneo

Probar 81 configuraciones con todos los datos es caro. Pero muchas son **tan malas** que se nota enseguida, incluso entrenando con pocos datos. *Successive halving* (reducción sucesiva a la mitad) organiza un **torneo**, como un casting de talentos:

```
Ronda 1:  81 candidatos  ×  pocos datos (p. ej. 100 filas)   → pasan los mejores 1/3
Ronda 2:  27 candidatos  ×  3 veces más datos (300 filas)    → pasan los mejores 1/3
Ronda 3:   9 candidatos  ×  3 veces más datos (900 filas)    → gana el mejor
```

El número que divide (aquí 3) es el hiperparámetro `factor`. Así se gasta poco en los malos y mucho en los prometedores. En scikit-learn: `HalvingGridSearchCV` y `HalvingRandomSearchCV` (todavía marcadas como experimentales, por eso la línea `enable_halving_search_cv` de la preparación).

### 🔮 Ejercicio 6 — Calcula el torneo
Con 81 candidatos y `factor=3`, si los datos alcanzaran para muchas rondas, ¿cuántos candidatos quedarían en cada ronda hasta tener un ganador?

✍️ *Mi cálculo:*

<details><summary><b>👀 Ver respuesta</b></summary>

81 → 27 → 9 → 3 → **1**: cinco rondas. En la práctica, el torneo se detiene antes si en la siguiente ronda ya no hay datos suficientes (cada ronda usa 3 veces más filas). Ahora veamos cuántas rondas caben con nuestros dígitos.
</details>

In [ ]:
inicio = time.time()
torneo = HalvingRandomSearchCV(SVC(), distribuciones, n_candidates=81, factor=3,
                               cv=5, n_jobs=-1, random_state=0).fit(X_d_tr, y_d_tr)
t_torneo = time.time() - inicio

for ronda, (n_cand, n_filas) in enumerate(zip(torneo.n_candidates_, torneo.n_resources_), start=1):
    print(f"Ronda {ronda}: {n_cand:>2} candidatos entrenados con {n_filas:>4} filas")
print(f"\nTorneo: mejor exactitud CV {torneo.best_score_:.3f} en {t_torneo:.1f} s")

inicio = time.time()
todos = RandomizedSearchCV(SVC(), distribuciones, n_iter=81, cv=5, n_jobs=-1, random_state=0).fit(X_d_tr, y_d_tr)
print(f"Los 81 con todos los datos: mejor exactitud CV {todos.best_score_:.3f} en {time.time() - inicio:.1f} s")

✋ Compara el tiempo y la exactitud del torneo con los de evaluar los 81 candidatos con todos los datos. ¿Qué ganas y qué arriesgas?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- Caben **3 rondas**: 81 candidatos con 100 filas → 27 con 300 → 9 con 900. (Una cuarta ronda necesitaría 2,700 filas y solo hay 1,347).
- El torneo tardó **varias veces menos** que evaluar los 81 a fondo, y su mejor configuración tiene una exactitud CV ≈ 0.98, contra ≈ 0.99.

**Ganas:** muchísimo tiempo, sobre todo con muchos candidatos o datos grandes. **Arriesgas:** que un candidato que rinde mal con pocos datos pero muy bien con muchos sea eliminado en las primeras rondas. En la práctica, el ahorro casi siempre compensa.

> 📝 La `best_score_` del torneo se mide en la **última ronda** (con 900 filas), así que no es exactamente comparable con la de los 81 completos (con todas las filas).
</details>

---
## 5. 📖 Búsqueda bayesiana: el geólogo que aprende

Las estrategias anteriores tienen algo en común: **no aprenden** de los pozos que ya perforaron. La cuadrícula y el azar deciden todos los pozos de antemano.

La **búsqueda bayesiana** hace lo que haría un geólogo:
1. Perfora unos pocos pozos al azar.
2. Con esos resultados, dibuja un **mapa aproximado** del terreno: dónde cree que hay petróleo y **qué tan seguro** está (un *modelo sustituto*).
3. Elige el siguiente pozo equilibrando dos impulsos:
   - **Explotar:** perforar cerca de donde ya encontró mucho petróleo.
   - **Explorar:** perforar donde el mapa es muy **incierto**, por si hay algo mejor.
4. Actualiza el mapa con el nuevo resultado y repite.

Es el mismo dilema de **exploración vs. explotación** que verás en aprendizaje por refuerzo (Lección 14.2). Como cada pozo se elige con inteligencia, suele encontrar buenas configuraciones con **menos intentos** que el azar, sobre todo cuando cada entrenamiento es caro (minutos u horas).

La librería más popular es **Optuna**. Así se ve una búsqueda con ella:

```python
# ⚠️ Para Colab: Optuna no viene en este entorno. En Colab, primero ejecuta:  !pip install optuna
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

def objetivo(trial):
    # Optuna "propone" valores; tú solo describes el rango de cada hiperparámetro
    C     = trial.suggest_float("C", 1e-2, 1e2, log=True)
    gamma = trial.suggest_float("gamma", 1e-5, 1e-1, log=True)
    modelo = SVC(C=C, gamma=gamma)
    return cross_val_score(modelo, X_d_tr, y_d_tr, cv=5).mean()   # lo que quiere maximizar

estudio = optuna.create_study(direction="maximize")
estudio.optimize(objetivo, n_trials=20)          # el presupuesto: 20 pozos
print(estudio.best_params, estudio.best_value)

# Extras útiles de Optuna:
# - optuna.visualization.plot_optimization_history(estudio)   → cómo mejora el récord
# - optuna.importance.get_param_importances(estudio)          → qué hiperparámetros importaron (Lección 7.4)
# - "pruning": abandona a mitad de camino los intentos que van mal (idea parecida al torneo)
```

> 💡 Otras opciones con la misma idea: `BayesSearchCV` de *scikit-optimize* (se usa igual que `GridSearchCV`) y *Hyperopt*.

### ✋ Ejercicio 7 — ¿Qué estrategia usarías?
Para cada caso, elige entre cuadrícula, aleatoria, torneo (*halving*) o bayesiana:
1. Una regresión logística con **un solo** hiperparámetro (`C`) y datos pequeños.
2. Un gradient boosting con **6 hiperparámetros**, 2 millones de filas y una tarde de cómputo.
3. Una red neuronal que tarda **3 horas** en entrenarse; solo puedes permitirte unos 30 intentos.
4. Un Random Forest con 4 hiperparámetros y datos medianos; quieres algo rápido y sin instalar nada.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Cuadrícula**: con un solo hiperparámetro, 10–15 valores en escala log cubren todo. (`LogisticRegressionCV` lo hace por ti).
2. **Torneo (*halving*)**: con tantos datos, descartar candidatos malos usando pocas filas ahorra muchísimo tiempo. También es buena opción la bayesiana con *pruning*.
3. **Bayesiana**: cada intento es carísimo, así que conviene que cada pozo se elija aprendiendo de los anteriores.
4. **Aleatoria**: simple, controlas el presupuesto con `n_iter` y, con 4 hiperparámetros, explora mejor que una cuadrícula.

No hay una respuesta única; lo importante es razonar con **presupuesto** y **número de hiperparámetros**.
</details>

### 📋 Resumen comparativo

| Estrategia | ✅ Ventajas | ❌ Desventajas | Úsala cuando… |
|---|---|---|---|
| 🔲 Cuadrícula (`GridSearchCV`) | Simple, exhaustiva, reproducible | Explosión combinatoria; desperdicia intentos en hiperparámetros irrelevantes | 1–2 hiperparámetros, o para **refinar** alrededor de un buen punto |
| 🎲 Aleatoria (`RandomizedSearchCV`) | Presupuesto fijo; explora mejor lo que importa | Puede dejar zonas sin explorar por mala suerte | **El punto de partida por defecto** con 3 o más hiperparámetros |
| 🏆 Torneo (`HalvingRandomSearchCV`) | Muy rápida con muchos candidatos o datos grandes | Puede eliminar pronto a un candidato que solo brilla con muchos datos | Muchos candidatos y/o muchos datos |
| 🧠 Bayesiana (Optuna) | Necesita menos intentos; aprende de los anteriores | Librería extra; cada intento depende del anterior (menos paralelizable) | Cada entrenamiento es **caro** (minutos u horas) |

---
## 6. 📖 Errores comunes: `best_score_` es optimista

Este es el error más traicionero de toda la lección. Cuando pruebas muchas configuraciones y te quedas con la **mejor**, parte de su buen puntaje es **suerte**: entre muchos intentos, alguno sale bien "de rebote". Es como lanzar 50 monedas y declarar "experta en caras" a la que sacó más caras.

### 🔮 Ejercicio 8 — Buscar en datos sin señal
Creamos datos **totalmente aleatorios**: las etiquetas son monedas al aire, sin ninguna relación con las características. Ningún modelo puede superar el **50 %** de exactitud con datos nuevos.

Hacemos una búsqueda aleatoria de 50 configuraciones de un árbol de decisión. ¿Qué `best_score_` crees que reportará? ¿Y en la prueba?

✍️ *Mi predicción:*

In [ ]:
rng = np.random.default_rng(0)
X_ruido = rng.normal(size=(200, 10))
y_ruido = rng.integers(0, 2, 200)            # monedas al aire: no hay nada que aprender
X_r_tr, X_r_te, y_r_tr, y_r_te = train_test_split(X_ruido, y_ruido, test_size=0.5, random_state=0)

busqueda = RandomizedSearchCV(DecisionTreeClassifier(random_state=0),
                              {"max_depth": randint(1, 15), "min_samples_leaf": randint(1, 30),
                               "criterion": ["gini", "entropy"]},
                              n_iter=50, cv=5, random_state=0).fit(X_r_tr, y_r_tr)
print(f"best_score_ (validación cruzada del ganador): {busqueda.best_score_:.1%}")
print(f"Exactitud en prueba del ganador            : {busqueda.score(X_r_te, y_r_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

La búsqueda reporta un `best_score_` de ≈ **61 %**, ¡con datos que no tienen ninguna señal! En la prueba, el ganador cae a ≈ **40 %**: no es mejor que lanzar una moneda (con solo 100 ejemplos de prueba, caer un poco por debajo del 50 % también es cuestión de suerte).

El 61 % no es mentira: es el mejor de 50 intentos, y **el mejor de muchos intentos siempre está inflado por la suerte**. Por eso:
1. **Nunca** reportes `best_score_` como el desempeño esperado del modelo.
2. Guarda un **conjunto de prueba** que la búsqueda no toque y evalúa ahí **una sola vez** al final.
3. Si no tienes datos para un conjunto de prueba, usa **validación cruzada anidada** (Lección 7.5).

Cuantas **más** configuraciones pruebes, más se infla el mejor puntaje. Es otra razón para no buscar sin límite.
</details>

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El modelo rinde peor en prueba que su `best_score_` | Optimismo por selección (Ejercicio 8) | Reportar la prueba, no `best_score_`; validación cruzada anidada |
| El mejor valor está en el extremo del rango | El rango era demasiado estrecho | Ampliar el rango en esa dirección |
| "Probé `C` = 1, 2, 3, 4, 5 y todos dan igual" | Escala lineal para una perilla logarítmica | `loguniform` o listas tipo 0.01, 0.1, 1, 10 |
| La búsqueda tarda horas | Cuadrícula con muchos hiperparámetros | Aleatoria con `n_iter` fijo, o *halving* |
| Resultados distintos cada vez que ejecutas | Sin semillas | Fijar `random_state` en la búsqueda y en el modelo |
| El "mejor" modelo es malo para el negocio | Se optimizó la métrica equivocada (p. ej. exactitud con clases desbalanceadas) | Elegir bien `scoring=` (Lección 4.8) |

---
## 7. 🧠 Resumen en 5 puntos
1. Buscar hiperparámetros es **gastar un presupuesto** de entrenamientos; cada configuración se evalúa con validación cruzada.
2. La **cuadrícula** prueba todo, pero su costo explota con cada hiperparámetro nuevo y desperdicia intentos en los irrelevantes.
3. La **aleatoria** explora mejor lo que importa con el mismo presupuesto: es el punto de partida por defecto. Usa escala **logarítmica** para perillas como `C`, `alpha` o `learning_rate`.
4. El **torneo** (*successive halving*) descarta pronto a los malos usando pocos datos; la **bayesiana** (Optuna) aprende de los intentos anteriores y conviene cuando cada entrenamiento es caro.
5. `best_score_` es **optimista**: el resultado honesto sale de un conjunto de prueba que la búsqueda nunca vio.

---
## 8. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 8.1 Repaso rápido (opción múltiple)

**1.** Una cuadrícula con 3 hiperparámetros de 5 valores cada uno y validación cruzada de 5 pliegues entrena…
- a) 15 modelos · b) 75 modelos · c) 125 modelos · d) 625 modelos

**2.** ¿Por qué la búsqueda aleatoria suele ganar a la cuadrícula con el mismo presupuesto?
- a) Porque usa más datos
- b) Porque prueba más valores distintos de los hiperparámetros que importan
- c) Porque es determinista
- d) Porque no usa validación cruzada

**3.** ¿Cuál es la mejor distribución para buscar `learning_rate` entre 0.001 y 0.5?
- a) `randint(0, 1)` · b) Una lista `[0.1, 0.2, 0.3, 0.4, 0.5]` · c) `loguniform(1e-3, 0.5)` · d) Siempre 0.1

**4.** En *successive halving* con `factor=3`, ¿qué pasa de una ronda a la siguiente?
- a) Quedan 3 candidatos · b) Queda 1/3 de los candidatos, con 3 veces más datos · c) Se triplican los candidatos · d) Se usan 3 pliegues

**5.** Tu búsqueda reporta `best_score_ = 0.91`. ¿Qué número le das a tu jefe como desempeño esperado?
- a) 0.91 · b) Algo más de 0.91 · c) El resultado en un conjunto de prueba que la búsqueda no vio · d) El promedio de todas las configuraciones

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **d)** 5 × 5 × 5 = 125 combinaciones × 5 pliegues = 625.
2. **b)** Con el mismo número de intentos, cada uno prueba un valor distinto de **cada** hiperparámetro.
3. **c)** Escala logarítmica: da la misma oportunidad a 0.001–0.01 que a 0.01–0.1 o a 0.1–0.5.
4. **b)** Es el torneo: menos candidatos, más recursos para cada uno.
5. **c)** `best_score_` está inflado por la selección (Ejercicio 8).
</details>

### 8.2 Preguntas para pensar (sin ejecutar)

**A.** Si la búsqueda aleatoria es tan buena, ¿tiene algún sentido usar todavía la cuadrícula? Da un ejemplo.

**B. Caso de análisis 💼** Un equipo te muestra este informe: *"Probamos 2,000 configuraciones de un gradient boosting con búsqueda en cuadrícula. La mejor alcanzó 0.947 de AUC en validación cruzada, frente a 0.941 de la configuración por defecto. Recomendamos la nueva configuración."* ¿Qué preguntas harías antes de aceptar la recomendación?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Sí: (1) cuando hay **uno o dos** hiperparámetros, una cuadrícula en escala log es simple y completa; (2) para **refinar**: tras una búsqueda aleatoria amplia, una cuadrícula pequeña alrededor del mejor punto; (3) cuando necesitas una tabla ordenada de resultados para un informe o un mapa de calor como el de la Sección 2.

**B.** Algunas preguntas clave:
- ¿La mejora (0.006) es mayor que el **ruido** de la validación cruzada? ¿Cuál es la desviación estándar entre pliegues? (Lección 7.4)
- Con **2,000** intentos, el mejor está muy inflado por la suerte (Ejercicio 8). ¿Cuánto da en un **conjunto de prueba** que nadie haya tocado?
- ¿Vale la pena el costo de cómputo? ¿Una búsqueda aleatoria con 60 intentos habría llegado a lo mismo?
- ¿El mejor valor de algún hiperparámetro quedó en el borde de la cuadrícula?
</details>

### 8.3 🧩 Práctica integradora: tres estrategias, un presupuesto

Una aseguradora quiere detectar **reclamaciones sospechosas**. Tienes 4,000 reclamaciones con 20 características y vas a ajustar un `HistGradientBoostingClassifier` con **4 hiperparámetros**. Compararás tres estrategias y elegirás con criterio.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Datos y línea base con los valores por defecto**

In [ ]:
X_s, y_s = make_classification(n_samples=4_000, n_features=20, n_informative=8, n_redundant=4,
                               flip_y=0.2, class_sep=0.8, random_state=1)
X_s_tr, X_s_te, y_s_tr, y_s_te = train_test_split(X_s, y_s, test_size=0.25, stratify=y_s, random_state=1)

base = HistGradientBoostingClassifier(random_state=0)
cv_base = cross_val_score(base, X_s_tr, y_s_tr, cv=5, scoring="roc_auc")
print(f"Línea base (valores por defecto): AUC CV = {cv_base.mean():.4f} ± {cv_base.std():.4f}")

**Paso 2 ▶️ — Define el espacio de búsqueda.** Lee cada línea: ¿por qué crees que `learning_rate` y `l2_regularization` usan `loguniform` y los otros `randint`?

In [ ]:
espacio = {
    "learning_rate":     loguniform(0.01, 0.5),   # tamaño del paso (Lección 7.2)
    "max_leaf_nodes":    randint(4, 64),          # complejidad de cada árbol
    "min_samples_leaf":  randint(5, 100),         # freno: tamaño mínimo de las hojas
    "l2_regularization": loguniform(1e-3, 10),    # cobro L2 (Lección 7.1)
}

<details><summary><b>👀 Ver respuesta</b></summary>

`learning_rate` y `l2_regularization` son perillas cuyo efecto se nota por **órdenes de magnitud** (0.01 vs. 0.1 vs. 1): escala logarítmica. `max_leaf_nodes` y `min_samples_leaf` son **conteos enteros** que se mueven en un rango moderado: `randint` está bien.
</details>

**Paso 3 🔮 — Aleatoria contra torneo.** Ejecutamos una búsqueda aleatoria con **20** candidatos y un torneo con **60** candidatos. ¿Cuál crees que tardará más? ¿Cuál encontrará mejor AUC?

✍️ *Mi apuesta:*

In [ ]:
resultados = {}

inicio = time.time()
aleatoria = RandomizedSearchCV(HistGradientBoostingClassifier(random_state=0), espacio, n_iter=20,
                               cv=5, scoring="roc_auc", n_jobs=-1, random_state=0).fit(X_s_tr, y_s_tr)
resultados["Aleatoria (20)"] = (aleatoria, time.time() - inicio)

inicio = time.time()
halving = HalvingRandomSearchCV(HistGradientBoostingClassifier(random_state=0), espacio, n_candidates=60,
                                factor=3, min_resources=300,   # la 1.ª ronda usa 300 filas
                                cv=5, scoring="roc_auc", n_jobs=-1, random_state=1).fit(X_s_tr, y_s_tr)
resultados["Torneo (60)"] = (halving, time.time() - inicio)

for nombre, (b, t) in resultados.items():
    print(f"{nombre:<15} AUC CV del ganador = {b.best_score_:.4f} · {t:5.1f} s")
print("\nRondas del torneo:", list(zip(halving.n_candidates_, halving.n_resources_)))

<details><summary><b>👀 Ver respuesta</b></summary>

Los dos tardan **del mismo orden**, aunque el torneo evaluó **el triple** de candidatos: a la mayoría solo los entrenó con 300 filas. Las rondas fueron 60 candidatos (300 filas) → 20 (900 filas) → 7 (2,700 filas).

En AUC, los ganadores empatan (≈ 0.866) y quedan algo por encima de la línea base (≈ 0.861). Ojo: la `best_score_` del torneo se mide en su última ronda (2,700 filas, no las 3,000 del entrenamiento), así que para comparar con justicia hay que reevaluar los ganadores en igualdad de condiciones (Paso 4).

> ⏱️ Los tiempos dependen mucho de tu computadora y de cuántos núcleos tenga. Fíjate en las proporciones, no en los segundos exactos.
</details>

**Paso 4 ✋ — ¿La mejora es real o es ruido?** Reevaluamos los dos ganadores con la **misma** validación cruzada que la línea base. Compara la mejora con la desviación estándar (±) de la línea base.

In [ ]:
for nombre, (b, _) in resultados.items():
    cv = cross_val_score(b.best_estimator_, X_s_tr, y_s_tr, cv=5, scoring="roc_auc")
    print(f"{nombre:<15} AUC CV = {cv.mean():.4f} ± {cv.std():.4f}   mejora vs. base: {cv.mean() - cv_base.mean():+.4f}")
print(f"{'Línea base':<15} AUC CV = {cv_base.mean():.4f} ± {cv_base.std():.4f}")

✍️ *Mi conclusión:*

<details><summary><b>👀 Ver respuesta</b></summary>

Los ganadores mejoran la línea base en apenas **+0.004 / +0.005** de AUC, mientras que la desviación estándar entre pliegues es ≈ **0.016**: ¡la mejora es **más pequeña que el ruido**! Con estos números no podemos afirmar que el modelo ajustado sea mejor de verdad.

No es un fracaso tuyo: los valores por defecto de `HistGradientBoosting` ya son buenos. Esto es muy habitual con los modelos modernos y lo estudiarás en la Lección 7.4: **ajustar hiperparámetros suele dar mejoras pequeñas**, y hay que compararlas siempre con el ruido de la validación cruzada.
</details>

**Paso 5 ▶️ — Examen final, una sola vez.** Elige un ganador y evalúalo en la prueba junto con la línea base.

In [ ]:
GANADOR = "Torneo (60)"     # ✏️ o "Aleatoria (20)"
from sklearn.metrics import roc_auc_score

final = resultados[GANADOR][0].best_estimator_      # ya viene reentrenado con todo el entrenamiento
base.fit(X_s_tr, y_s_tr)
print(f"{GANADOR}: AUC prueba = {roc_auc_score(y_s_te, final.predict_proba(X_s_te)[:, 1]):.4f}")
print(f"Línea base : AUC prueba = {roc_auc_score(y_s_te, base.predict_proba(X_s_te)[:, 1]):.4f}")
print("Hiperparámetros elegidos:", {k: (int(v) if float(v).is_integer() else round(float(v), 4))
                                  for k, v in resultados[GANADOR][0].best_params_.items()})

<details><summary><b>👀 Ver respuesta</b></summary>

En la prueba, el ganador del torneo logra ≈ **0.860** contra ≈ **0.853** de la línea base: una ventaja pequeña y en la misma dirección que en la validación cruzada. Conclusión honesta: la búsqueda ayudó **un poco**, quizá, y lo podemos decir con tranquilidad porque la prueba no participó en ninguna decisión. Si te hubieras quedado con el `best_score_` (0.866) como "desempeño esperado", habrías prometido más de lo que el modelo entrega.

Fíjate en los hiperparámetros elegidos: no hace falta que sean "bonitos". Lo que importa es que caigan dentro de la región buena.
</details>

---
## 📚 Para profundizar
- 📄 **Artículo:** Bergstra & Bengio (2012), *Random Search for Hyper-Parameter Optimization*: el origen de la figura de la Sección 3.
- 📘 **Documentación:** scikit-learn, guía de usuario, *Tuning the hyper-parameters of an estimator* (incluye *successive halving*).
- 🧪 **Optuna:** tutorial oficial en optuna.org (funciona directamente en Colab).

---
⬅️ **Anterior:** [7.2 Early stopping: saber cuándo parar](7.2-early-stopping.ipynb) · ➡️ **Siguiente:** [7.4 Cómo priorizar qué hiperparámetros ajustar primero](7.4-priorizar-hiperparametros.ipynb)